# Notebook 03 - Source Retrieval v1 POC


In [ ]:

# ============================================================
# Notebook 03 - Source Retrieval
# Agent Evaluation Framework v1 POC
# ============================================================

try:
    run_id
except NameError:
    run_id = "RUN-MANUAL-TEST"

try:
    environment
except NameError:
    environment = "dev"

try:
    poc_mode
except NameError:
    poc_mode = "false"
poc_mode = str(poc_mode).lower()

import datetime as dt
import hashlib
import re

import requests
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType

assert spark is not None, "Spark session not available."

LAKEHOUSE_NAME = "jacks_lakehouse"
ONELAKE_WORKSPACE_ID = "d9e51304-2b8a-4e62-b689-922e79fd76b4"
ONELAKE_LAKEHOUSE_ID = "537823c4-b83a-4a9b-9444-6039d55a4b9e"
LAKEHOUSE_DEFAULT_FILES_ROOT = "/lakehouse/default/Files"
LAKEHOUSE_FILES_ABFSS_ROOT = f"abfss://{ONELAKE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{ONELAKE_LAKEHOUSE_ID}/Files"
LAKEHOUSE_FILES_ROOT = LAKEHOUSE_FILES_ABFSS_ROOT
BASE_FILES_PATH = f"{LAKEHOUSE_FILES_ROOT}/agent_eval"
BASE_FILES_ABFSS_PATH = f"{LAKEHOUSE_FILES_ABFSS_ROOT}/agent_eval"

CURRENT_RUN_CASES_TABLE = "agent_eval_current_run_cases"
SOURCE_EVIDENCE_TABLE = "agent_eval_source_evidence"
PUBLIC_URL_TIMEOUT_SECONDS = 20
FABRIC_TABLE_IDENTIFIER_PATTERN = r"^[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*){0,2}$"


def now_utc():
    return dt.datetime.now(dt.timezone.utc)


def sha(text):
    return hashlib.sha256((text or "").encode("utf-8")).hexdigest()


def clean_html(html):
    text = re.sub(r"<script[\s\S]*?</script>", " ", html, flags=re.I)
    text = re.sub(r"<style[\s\S]*?</style>", " ", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def validate_fabric_table_identifier(table_name):
    if not re.fullmatch(FABRIC_TABLE_IDENTIFIER_PATTERN, table_name or ""):
        raise ValueError(f"Invalid fabric_table source_ref: {table_name}")
    return table_name


def fetch_source(case):
    source_type = (case.get("source_type") or "manual").strip()
    source_ref = case.get("source_ref") or ""
    if source_type == "manual":
        return True, "manual", source_ref, None, None
    if source_type == "public_url":
        response = requests.get(source_ref, timeout=PUBLIC_URL_TIMEOUT_SECONDS)
        response.raise_for_status()
        return True, f"HTTP {response.status_code}", clean_html(response.text)[:8000], response.headers.get("Last-Modified"), None
    if source_type == "fabric_table":
        source_ref = validate_fabric_table_identifier(source_ref)
        rows = spark.table(source_ref).limit(50).toJSON().collect()
        return True, f"fabric_table:{source_ref}", "\n".join(rows), None, None
    if source_type == "graph_sharepoint":
        return False, "GRAPH_NOT_CONFIGURED", "", None, "Graph retrieval needs Sites.Selected app registration and resolver details"
    return False, "UNKNOWN_SOURCE_TYPE", "", None, f"Unsupported source_type {source_type}"


def build_row(case):
    try:
        passed, status, excerpt, modified, error = fetch_source(case)
    except Exception as exc:
        passed, status, excerpt, modified, error = False, "ERROR", "", None, str(exc)[:500]
    return {
        "run_id": run_id,
        "test_id": case["test_id"],
        "agent_id": case["agent_id"],
        "source_type": case.get("source_type"),
        "source_ref": case.get("source_ref"),
        "retrieval_passed": bool(passed),
        "source_status": status,
        "raw_excerpt": excerpt,
        "version_hash": sha(excerpt),
        "retrieved_at": now_utc(),
        "modified_at": modified,
        "error_details": error,
    }


schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("test_id", StringType(), False),
    StructField("agent_id", StringType(), False),
    StructField("source_type", StringType(), True),
    StructField("source_ref", StringType(), True),
    StructField("retrieval_passed", BooleanType(), False),
    StructField("source_status", StringType(), True),
    StructField("raw_excerpt", StringType(), True),
    StructField("version_hash", StringType(), True),
    StructField("retrieved_at", TimestampType(), False),
    StructField("modified_at", StringType(), True),
    StructField("error_details", StringType(), True),
])

cases = [r.asDict() for r in spark.table(CURRENT_RUN_CASES_TABLE).filter(F.col("run_id") == run_id).collect()]
if not cases:
    raise RuntimeError("No current run cases found for Notebook 03")
rows = [build_row(case) for case in cases]
spark.createDataFrame([Row(**r) for r in rows], schema=schema).write.format("delta").mode("append").saveAsTable(SOURCE_EVIDENCE_TABLE)

failed = sum(1 for r in rows if not r["retrieval_passed"])
print(f"Source retrieval complete. rows={len(rows)} failed={failed}")
if failed == len(rows):
    raise RuntimeError("All source retrieval rows failed")

try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit("PASS")
except ImportError:
    pass
